In [ ]:
import osgeo.ogr
import osgeo.osr

FT2M = 0.3048
MSL_OFFSET = 34.7842

nad83 = osgeo.osr.SpatialReference()
nad83.ImportFromEPSG(2893)

ecef = osgeo.osr.SpatialReference()
ecef.ImportFromEPSG(4978)

def ftMSL2Ell(e):
    return e*FT2M - MSL_OFFSET

def distance(p0, p1):
    res = {
        "dx": p1.GetX() - p0.GetX(),
        "dy": p1.GetY() - p0.GetY(),
        "dz": p1.GetZ() - p0.GetZ(),
        "d": None
    }
    res["d"] = (res["dx"]**2 + res["dy"]**2 + res["dz"]**2)**0.5
    return res
        
    

In [21]:
transform = osgeo.osr.CoordinateTransformation(nad83, ecef)

benchmark = [1434482.29, 496145.72, ftMSL2Ell(9.69), 0]
bm_meas = [1434480.40440, 496147.88627, ftMSL2Ell(4.1), 0]

p0 = osgeo.ogr.Geometry(osgeo.ogr.wkbPoint)
p0.AddPointZM(*benchmark)
p1 = osgeo.ogr.Geometry(osgeo.ogr.wkbPoint)
p1.AddPointZM(*bm_meas)

print(distance(p0,p1))

p0.Transform(transform)
p1.Transform(transform)
print(distance(p0,p1))


{'dx': -1.8855999999213964, 'dy': 2.1662700000451878, 'dz': -1.703832000000002, 'd': 3.3393497206826592}
{'dx': -0.960389363579452, 'dy': 1.560717818327248, 'dz': -0.5578597183339298, 'd': 1.9155665750530106}


In [ ]:
import pyproj

#Latitude, longitude in degrees
lat = 39.02699686
lon = -76.56400495
#height above ellipsoid in meters
hae = 16.166

# This aligns with the information reported from rtk2go
transformer = pyproj.Transformer.from_crs(
    {"proj":'latlong', "ellps":'WGS84', "datum":'WGS84'},
    {"proj":'geocent', "ellps":'WGS84', "datum":'WGS84'},
    )
x ,y, z = transformer.transform(lon,lat,hae,radians = False)
print(x, y, z)

#from rtk2go MorjanFarms
lat = 44.30016061
lon = -79.14897320
hae = 249.324
x ,y, z = transformer.transform(lon,lat,hae,radians = False)
print(x, y, z)


1152838.244883493 -4825658.722604054 3994655.9236075096
860786.3984649417 -4490668.519163037 4432196.654311733


In [42]:
'''
There's a discrepancy between values calculated here and in QGIS
'''
import json
LL2NAD = pyproj.Transformer.from_crs("epsg:4326","epsg:2893") #"epsg:2248")
NAD2LL = pyproj.Transformer.from_crs("epsg:2893","epsg:4326")
NAD2ECEF = pyproj.Transformer.from_crs("epsg:2893", "epsg:4978")

def delta(p0, p1):
    dx = p1[0] - p0[0]
    dy = p1[1] - p0[1]
    dz = p1[2] - p0[2]
    return (dx**2 + dy**2 + dz**2)**0.5

def ll2nad(x, y, z):
    return LL2NAD.transform(x, y, z, radians=False)

def nad2ll(x, y, z):
    return NAD2LL.transform(x, y, z, radians=False)

bm = {
    "p":[1434482.09933, 496145.93646, ftMSL2Ell(9.69)],
    "ll_ex": [39.02821081, -76.57001919, ftMSL2Ell(9.69)],
    "ll": None,
}

meas = {
    "p": [1434480.40668, 496147.89077, ftMSL2Ell(4.1)],
    "ll_ex": [39.02821616, -76.57002519, ftMSL2Ell(4.1)],
    "ll": None,
}

bm["ll"] = nad2ll(*bm['p'])
bm['ll_d'] = delta(bm['ll_ex'], bm['ll'])
bm['inv'] = ll2nad(*bm['ll'])
bm['inv_d'] = delta(bm['p'], bm['inv'])
print(json.dumps(bm, indent=2))


{
  "p": [
    1434482.09933,
    496145.93646,
    -31.830688
  ],
  "ll_ex": [
    39.02821081,
    -76.57001919,
    -31.830688
  ],
  "ll": [
    39.02821860594389,
    -76.57002125025983,
    -31.830688
  ],
  "ll_d": 8.063585540583787e-06,
  "inv": [
    1434482.0993295796,
    496145.9364611142,
    -31.830688
  ],
  "inv_d": 1.1909155701018384e-06
}


In [102]:
class PointXYZ:
    def __init__(self, x:float, y:float, z:float):
        self.x = x
        self.y = y
        self.z = z

    def __str__(self):
        return json.dumps({
            'x': self.x,
            'y': self.y,
            'z': self.z
        }, indent=2)
        
        
class Point4326(PointXYZ):
    '''3D Point in EPSG:4326 (WGS84)
    x = longitude degrees
    y = latitude degrees
    z = ellipsoid elevation in meters
    '''
    def __init__(self, x, y, z):
        super().__init__(x,y,z)

    def __str__(self):
        return json.dumps({
            'longitude': self.x,
            'latitude': self.y,
            'elev_e': self.z
        }, indent=2)
        
class Point4978(PointXYZ):
    '''3D Point in EPSG4978 (ECEF, GRS80)
    
    All dimensions in meters.
    '''
    def __init__(self, x, y, z):
        super().__init__(x,y,z)

    def __str__(self):
        return json.dumps({
            'x': self.x,
            'y': self.y,
            'z': self.z
        }, indent=2)
        

class Point2893(PointXYZ):
    '''3D point in EPSG:2893.
    
    x = easting in feet
    y = northing in feet
    z = MSL elevation in feet
    '''
    def __init__(self, x, y, z):
        super().__init__(x,y,z)
        
    @property
    def zm(self):
        '''MSL Z in meters
        '''
        return self.z * FT2M
    
    @property
    def zme(self):
        '''Ellipsoid Z in meters
        '''
        return (self.z * FT2M)-MSL_OFFSET
    
    @property
    def llz(self)->Point4326:
        '''Return WGS84 longitude, latitude, ellipsoid elevation (m)
        '''
        _ll = NAD2LL.transform(self.x, self.y, self.zme, radians=False)
        return Point4326(_ll[1], _ll[0], _ll[2])
    
    @property
    def ecef(self)->Point4978:
        _ecef = NAD2ECEF.transform(self.x, self.y, self.zme, radians=False) 
        return Point4978(*_ecef)

    def __str__(self):
        _llz = self.llz
        _ec = self.ecef
        _d = {
            'x': self.x, 'y': self.y, 'z': self.z,
            'llz': {'lon':_llz.x, 'lat':_llz.y, 'eleve': _llz.z},
            'ecef':{'x':_ec.x, 'y':_ec.y, 'z':_ec.z},
        }
        return json.dumps(_d, indent=2)
            
    
    def dxyz(self, p1:'Point2893'):
        '''Delta x,y,z from self to p1
        '''
        return (
            p1.x - self.x,
            p1.y - self.y,
            p1.z - self.z
        )
    
    def deltaFt(self, p1:'Point2893'):
        '''Distance in ft from self to p1
        '''
        d = self.dxyz(p1)
        return (d[0]**2 + d[1]**2 + d[2]**2)**0.5
    
    def deltaM(self, p1:'Point2893'):
        a = self.ecef
        b = p1.ecef
        return ((b.x-a.x)**2 + (b.y-a.y)**2 + (b.z - a.z)**2)**0.5

    def offset(self, delta):        
        '''New point offset delta feet x,y,z from self.
        '''
        p = Point2893(self.x + delta[0], self.y + delta[1], self.z + delta[2])
        return p

    
def Point2893FromLL(lon, lat, ft_msl):
    x,y,z = LL2NAD.transform(lat, lon, ft_msl)
    return Point2893(x, y, z)
    

In [103]:

benchmark1 = Point2893(1434482.29, 496145.72, 9.69)
benchmark2 = Point2893(1434386.39, 496060.63, 23)
benchmark3 = Point2893(1436430.18, 495644.27, 106)

print(benchmark1)

{
  "x": 1434482.29,
  "y": 496145.72,
  "z": 9.69,
  "llz": {
    "lon": -76.57002058267335,
    "lat": 39.028218009153974,
    "eleve": -31.830688
  },
  "ecef": {
    "x": 1152303.0933983384,
    "y": -4825660.437793732,
    "z": 3994731.0145210037
  }
}


In [104]:
measured = Point2893(1434480.40411, 496147.94697, 5.6)
print(measured)

{
  "x": 1434480.40411,
  "y": 496147.94697,
  "z": 5.6,
  "llz": {
    "lon": -76.57002718425258,
    "lat": 39.028224148033416,
    "eleve": -33.07732
  },
  "ecef": {
    "x": 1152302.2127922724,
    "y": -4825659.211200943,
    "z": 3994730.7589347763
  }
}


In [105]:
print(benchmark1.deltaFt(measured))
print(benchmark1.dxyz(measured))

5.024348363042558
(-1.8858900000341237, 2.2269700000179, -4.09)


In [106]:
base = Point2893FromLL(-76.56400495, 39.02699686, (16.166+MSL_OFFSET)/FT2M)
print(base)

{
  "x": 1436193.3191949364,
  "y": 495709.0784843127,
  "z": 167.1594488188976,
  "llz": {
    "lon": -76.56400495014373,
    "lat": 39.02699686036875,
    "eleve": 16.16599999999999
  },
  "ecef": {
    "x": 1152838.2448653975,
    "y": -4825658.722581875,
    "z": 3994655.923639312
  }
}


In [107]:
base_offset = base.offset(measured.dxyz(benchmark1))
print(base_offset)
print(base.deltaM(base_offset))
ecef = base_offset.ecef

{
  "x": 1436195.2050849365,
  "y": 495706.8515143127,
  "z": 171.2494488188976,
  "llz": {
    "lon": -76.56399834919749,
    "lat": 39.02699072114798,
    "eleve": 17.412631999999995
  },
  "ecef": {
    "x": 1152839.1255685387,
    "y": -4825659.949123186,
    "z": 3994656.179162178
  }
}
1.531448186161642
